In [ ]:
import re
import pandas as pd

# Basit token sayacı
def count_tokens(text):
    if pd.isna(text):
        return 0
    text = str(text).strip()
    tokens = re.findall(r"\b\w+\b", text)
    return len(tokens)

# Karakter sayacı
def count_chars(text):
    if pd.isna(text):
        return 0
    return len(str(text).strip())

In [ ]:
from google.colab import files

uploaded = files.upload()

Saving npc_dataset.jsonl to npc_dataset.jsonl


In [ ]:
import json
import pandas as pd

DATA_PATH = "npc_dataset.jsonl"

records = []

with open(DATA_PATH, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            records.append(json.loads(line))

print("Toplam kayıt sayısı:", len(records))
print(records[0])

Toplam kayıt sayısı: 91720
{'id': 1, 'domain_id': 'EXP_PHY_MEC_001', 'domain': 'physics', 'npc': {'role': 'virtual_physics_teacher', 'role_style': 'explanatory', 'roleplay_style': 'supportive'}, 'dialogue': {'question_tr': 'Cisimlerin hareketini değiştiren etki ne ad alır?', 'question_en': 'What is the name given to the effect that changes the motion of objects?', 'answer_tr': 'Kuvvet, bir cismin hareketini veya şeklini değiştirebilen etkidir.', 'answer_en': 'Force is an effect that can change the motion or shape of an object.'}, 'labels': {'intent': 'CONCEPT_EXPLAIN', 'player_intent': 'ask_definition', 'emotion': 'informative', 'game_context': 'tutorial', 'difficulty_level': 'very_easy'}, 'tags': {'tr': ['mekanik', 'temel_kavram'], 'en': ['mechanic', 'basic_concept']}}


In [ ]:
df = pd.json_normalize(records)

print("Veri seti boyutu:", df.shape)
print("Kolonlar:")
print(df.columns.tolist())

df.head()

Veri seti boyutu: (91720, 17)
Kolonlar:
['id', 'domain_id', 'domain', 'npc.role', 'npc.role_style', 'npc.roleplay_style', 'dialogue.question_tr', 'dialogue.question_en', 'dialogue.answer_tr', 'dialogue.answer_en', 'labels.intent', 'labels.player_intent', 'labels.emotion', 'labels.game_context', 'labels.difficulty_level', 'tags.tr', 'tags.en']


,id,domain_id,domain,npc.role,npc.role_style,npc.roleplay_style,dialogue.question_tr,dialogue.question_en,dialogue.answer_tr,dialogue.answer_en,labels.intent,labels.player_intent,labels.emotion,labels.game_context,labels.difficulty_level,tags.tr,tags.en
0,1,EXP_PHY_MEC_001,physics,virtual_physics_teacher,explanatory,supportive,Cisimlerin hareketini değiştiren etki ne ad alır?,What is the name given to the effect that chan...,"Kuvvet, bir cismin hareketini veya şeklini değ...",Force is an effect that can change the motion ...,CONCEPT_EXPLAIN,ask_definition,informative,tutorial,very_easy,"[mekanik, temel_kavram]","[mechanic, basic_concept]"
1,2,EXP_PHY_MEC_002,physics,virtual_physics_teacher,explanatory,supportive,Kuvvet neyi belirler?,What determines the clear force?,Kuvvet cismin ivmesini belirler.,The clear force determines the acceleration of...,CONCEPT_EXPLAIN,ask_definition,informative,tutorial,very_easy,"[mekanik, kuvvet]","[mechanical, force]"
2,3,EXP_PHY_MEC_003,physics,virtual_physics_teacher,explanatory,supportive,Newton’un birinci yasası neyi açıklar?,What does Newton's first law explain?,Eylemsizlik ilkesini açıklar.,It explains the principle of inertia.,CONCEPT_EXPLAIN,ask_definition,informative,tutorial,very_easy,"[newton, hareket]","[newton, motion]"
3,4,EXP_PHY_MEC_004,physics,virtual_physics_teacher,explanatory,supportive,Newton’un ikinci yasası nedir?,What is Newton's second law?,"Kuvvet, kütle ve ivme arasındaki ilişkiyi açık...","It explains the relationship between force, ma...",CONCEPT_EXPLAIN,ask_information,informative,tutorial,easy,"[newton, ivme]","[newton, acceleration]"
4,5,EXP_PHY_MEC_005,physics,virtual_physics_teacher,explanatory,supportive,Newton’un üçüncü yasası neyi ifade eder?,What does Newton's third law state?,Etki–tepki kuvvetlerini ifade eder.,It expresses action and reaction forces.,CONCEPT_EXPLAIN,ask_information,informative,tutorial,easy,"[newton, etki_tepki]","[newton, action_response]"


In [ ]:
required_cols = [
    "id",
    "domain_id",
    "domain",
    "npc.role",
    "npc.role_style",
    "npc.roleplay_style",
    "dialogue.question_tr",
    "dialogue.answer_tr",
    "dialogue.question_en",
    "dialogue.answer_en",
    "labels.intent",
    "labels.player_intent",
    "labels.emotion",
    "labels.game_context",
    "labels.difficulty_level"
]

missing_cols = [col for col in required_cols if col not in df.columns]
print("Eksik kolonlar:", missing_cols)

if len(missing_cols) == 0:
    df = df[required_cols].copy()

    for col in df.columns:
        if df[col].dtype == "object":
            df[col] = df[col].astype(str).str.strip()

    df["domain"] = df["domain"].replace({
        "metaverse_ınformation": "metaverse_information"
    })

    df = df[
        (df["dialogue.question_en"].notna()) &
        (df["dialogue.answer_en"].notna()) &
        (df["dialogue.question_en"].str.len() > 2) &
        (df["dialogue.answer_en"].str.len() > 2)
    ].copy()

    print("Temizlik sonrası kayıt:", len(df))
    display(df.head())
else:
    print("Eksik kolonlar var. Önce kolon adlarını kontrol edin.")

Eksik kolonlar: []
Temizlik sonrası kayıt: 91675


,id,domain_id,domain,npc.role,npc.role_style,npc.roleplay_style,dialogue.question_tr,dialogue.answer_tr,dialogue.question_en,dialogue.answer_en,labels.intent,labels.player_intent,labels.emotion,labels.game_context,labels.difficulty_level
0,1,EXP_PHY_MEC_001,physics,virtual_physics_teacher,explanatory,supportive,Cisimlerin hareketini değiştiren etki ne ad alır?,"Kuvvet, bir cismin hareketini veya şeklini değ...",What is the name given to the effect that chan...,Force is an effect that can change the motion ...,CONCEPT_EXPLAIN,ask_definition,informative,tutorial,very_easy
1,2,EXP_PHY_MEC_002,physics,virtual_physics_teacher,explanatory,supportive,Kuvvet neyi belirler?,Kuvvet cismin ivmesini belirler.,What determines the clear force?,The clear force determines the acceleration of...,CONCEPT_EXPLAIN,ask_definition,informative,tutorial,very_easy
2,3,EXP_PHY_MEC_003,physics,virtual_physics_teacher,explanatory,supportive,Newton’un birinci yasası neyi açıklar?,Eylemsizlik ilkesini açıklar.,What does Newton's first law explain?,It explains the principle of inertia.,CONCEPT_EXPLAIN,ask_definition,informative,tutorial,very_easy
3,4,EXP_PHY_MEC_004,physics,virtual_physics_teacher,explanatory,supportive,Newton’un ikinci yasası nedir?,"Kuvvet, kütle ve ivme arasındaki ilişkiyi açık...",What is Newton's second law?,"It explains the relationship between force, ma...",CONCEPT_EXPLAIN,ask_information,informative,tutorial,easy
4,5,EXP_PHY_MEC_005,physics,virtual_physics_teacher,explanatory,supportive,Newton’un üçüncü yasası neyi ifade eder?,Etki–tepki kuvvetlerini ifade eder.,What does Newton's third law state?,It expresses action and reaction forces.,CONCEPT_EXPLAIN,ask_information,informative,tutorial,easy


In [ ]:
import re

def count_tokens(text):
    if pd.isna(text):
        return 0
    text = str(text).strip()
    tokens = re.findall(r"\b\w+\b", text)
    return len(tokens)

def count_chars(text):
    if pd.isna(text):
        return 0
    return len(str(text).strip())


# Türkçe uzunluklar
df["question_tr_token_len"] = df["dialogue.question_tr"].apply(count_tokens)
df["answer_tr_token_len"] = df["dialogue.answer_tr"].apply(count_tokens)
df["dialogue_tr_token_len"] = df["question_tr_token_len"] + df["answer_tr_token_len"]

df["question_tr_char_len"] = df["dialogue.question_tr"].apply(count_chars)
df["answer_tr_char_len"] = df["dialogue.answer_tr"].apply(count_chars)
df["dialogue_tr_char_len"] = df["question_tr_char_len"] + df["answer_tr_char_len"]


# İngilizce uzunluklar
df["question_en_token_len"] = df["dialogue.question_en"].apply(count_tokens)
df["answer_en_token_len"] = df["dialogue.answer_en"].apply(count_tokens)
df["dialogue_en_token_len"] = df["question_en_token_len"] + df["answer_en_token_len"]

df["question_en_char_len"] = df["dialogue.question_en"].apply(count_chars)
df["answer_en_char_len"] = df["dialogue.answer_en"].apply(count_chars)
df["dialogue_en_char_len"] = df["question_en_char_len"] + df["answer_en_char_len"]

df[
    [
        "dialogue.question_en",
        "dialogue.answer_en",
        "question_en_token_len",
        "answer_en_token_len",
        "dialogue_en_token_len"
    ]
].head()

,dialogue.question_en,dialogue.answer_en,question_en_token_len,answer_en_token_len,dialogue_en_token_len
0,What is the name given to the effect that chan...,Force is an effect that can change the motion ...,14,14,28
1,What determines the clear force?,The clear force determines the acceleration of...,5,9,14
2,What does Newton's first law explain?,It explains the principle of inertia.,7,6,13
3,What is Newton's second law?,"It explains the relationship between force, ma...",6,9,15
4,What does Newton's third law state?,It expresses action and reaction forces.,7,6,13


In [ ]:
length_stats = pd.DataFrame({
    "Metric": [
        "Avg. Turkish question length",
        "Avg. Turkish answer length",
        "Avg. Turkish dialogue length",
        "Max. Turkish dialogue length",
        "Avg. English question length",
        "Avg. English answer length",
        "Avg. English dialogue length",
        "Max. English dialogue length"
    ],
    "Token-based Value": [
        df["question_tr_token_len"].mean(),
        df["answer_tr_token_len"].mean(),
        df["dialogue_tr_token_len"].mean(),
        df["dialogue_tr_token_len"].max(),
        df["question_en_token_len"].mean(),
        df["answer_en_token_len"].mean(),
        df["dialogue_en_token_len"].mean(),
        df["dialogue_en_token_len"].max()
    ]
})

length_stats["Token-based Value"] = length_stats["Token-based Value"].round(2)
length_stats

,Metric,Token-based Value
0,Avg. Turkish question length,6.13
1,Avg. Turkish answer length,5.40
2,Avg. Turkish dialogue length,11.52
3,Max. Turkish dialogue length,46.00
4,Avg. English question length,8.62
5,Avg. English answer length,7.92
6,Avg. English dialogue length,16.54
7,Max. English dialogue length,64.00


In [ ]:
domain_length_stats = df.groupby("domain").agg(
    records=("id", "count"),
    avg_question_en_len=("question_en_token_len", "mean"),
    avg_answer_en_len=("answer_en_token_len", "mean"),
    avg_dialogue_en_len=("dialogue_en_token_len", "mean"),
    max_dialogue_en_len=("dialogue_en_token_len", "max")
).reset_index()

domain_length_stats = domain_length_stats.round(2)
domain_length_stats

,domain,records,avg_question_en_len,avg_answer_en_len,avg_dialogue_en_len,max_dialogue_en_len
0,academic,650,6.49,8.12,14.60,30
1,artificial_intelligence,1108,6.34,8.82,15.16,34
2,assessment,537,6.78,6.24,13.02,30
3,biology,457,6.97,8.89,15.86,33
4,chemistry,495,7.29,6.69,13.98,42
5,coding,458,7.22,8.13,15.36,36
6,counseling,750,7.05,8.09,15.15,25
7,crisis,1200,7.52,8.04,15.56,30
8,data_analysis,516,6.65,7.50,14.15,30
9,design_production,516,8.19,10.27,18.46,38


In [ ]:
# Excel dosyası olarak kaydetme
excel_path = "npc_dataset_length_statistics.xlsx"

with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
    # Genel uzunluk istatistikleri
    length_stats.to_excel(writer, sheet_name="General_Length_Stats", index=False)

    # Domain bazlı uzunluk istatistikleri
    domain_length_stats.to_excel(writer, sheet_name="Domain_Length_Stats", index=False)

    # Tüm veri seti + hesaplanan uzunluk kolonları
    df.to_excel(writer, sheet_name="Dataset_With_Lengths", index=False)

print("Excel dosyası oluşturuldu:", excel_path)

Excel dosyası oluşturuldu: npc_dataset_length_statistics.xlsx


In [ ]:
from google.colab import files

files.download("npc_dataset_length_statistics.xlsx")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [13]:
basic_stats = pd.DataFrame({
    "Metric": [
        "Total records",
        "Number of domains",
        "Number of domain IDs",
        "Number of NPC roles",
        "Number of role styles",
        "Number of roleplay styles",
        "Number of intent labels",
        "Number of player intent labels",
        "Number of emotion labels",
        "Number of game context labels",
        "Number of difficulty levels"
    ],
    "Value": [
        len(df),
        df["domain"].nunique(),
        df["domain_id"].nunique(),
        df["npc.role"].nunique(),
        df["npc.role_style"].nunique(),
        df["npc.roleplay_style"].nunique(),
        df["labels.intent"].nunique(),
        df["labels.player_intent"].nunique(),
        df["labels.emotion"].nunique(),
        df["labels.game_context"].nunique(),
        df["labels.difficulty_level"].nunique()
    ]
})

basic_stats

,Metric,Value
0,Total records,91675
1,Number of domains,32
2,Number of domain IDs,91675
3,Number of NPC roles,76
4,Number of role styles,36
5,Number of roleplay styles,36
6,Number of intent labels,37
7,Number of player intent labels,25
8,Number of emotion labels,40
9,Number of game context labels,30


In [14]:
domain_stats = df["domain"].value_counts().reset_index()
domain_stats.columns = ["Domain", "Records"]
domain_stats["Percentage"] = (domain_stats["Records"] / len(df) * 100).round(2)

domain_stats

,Domain,Records,Percentage
0,dialogues,20977,22.88
1,travel,20073,21.90
2,metaverse_information,18737,20.44
3,history_npc,12967,14.14
4,school_rules,2207,2.41
5,disease,1234,1.35
6,crisis,1200,1.31
7,artificial_intelligence,1108,1.21
8,family,1100,1.20
9,counseling,750,0.82


In [15]:
npc_role_stats = df["npc.role"].value_counts().reset_index()
npc_role_stats.columns = ["NPC Role", "Records"]
npc_role_stats["Percentage"] = (npc_role_stats["Records"] / len(df) * 100).round(2)

npc_role_stats.head(30)

,NPC Role,Records,Percentage
0,travel_advisor_npc,20073,21.90
1,dialogue_entertainment_npc,9296,10.14
2,information_expert,4000,4.36
3,dialogue_coach_npc,3352,3.66
4,dialogue_routine_npc,1908,2.08
5,dialogue_reflection_npc,1900,2.07
6,metaverse_guide_npc,1766,1.93
7,alparslan_npc,1480,1.61
8,metaverse_security_npc,1350,1.47
9,metaverse_vr_map_expert,1250,1.36


In [16]:
intent_stats = df["labels.intent"].value_counts().reset_index()
intent_stats.columns = ["Intent", "Records"]
intent_stats["Percentage"] = (intent_stats["Records"] / len(df) * 100).round(2)

intent_stats.head(30)

,Intent,Records,Percentage
0,ACADEMIC_OVERVIEW,27591,30.10
1,CONCEPT_EXPLAIN,17932,19.56
2,PROCESS_GUIDE,8504,9.28
3,REFLECTION,5505,6.00
4,KNOWLEDGE_CHECK,4262,4.65
5,AWARENESS_BUILD,4228,4.61
6,EVALUATE,4070,4.44
7,DECISION_SUPPORT,3918,4.27
8,LEARNING_SUPPORT,3866,4.22
9,DATA_COLLECT,1149,1.25


In [17]:
player_intent_stats = df["labels.player_intent"].value_counts().reset_index()
player_intent_stats.columns = ["Player Intent", "Records"]
player_intent_stats["Percentage"] = (player_intent_stats["Records"] / len(df) * 100).round(2)

player_intent_stats

,Player Intent,Records,Percentage
0,ask_information,51818,56.52
1,express_emotion,5082,5.54
2,make_choice,5011,5.47
3,ask_instruction,4939,5.39
4,ask_strategy,4059,4.43
5,ask_definition,2717,2.96
6,summarize_session,2651,2.89
7,request_help,2485,2.71
8,report_problem,2119,2.31
9,ask_direction,1979,2.16


In [18]:
emotion_stats = df["labels.emotion"].value_counts().reset_index()
emotion_stats.columns = ["Emotion", "Records"]
emotion_stats["Percentage"] = (emotion_stats["Records"] / len(df) * 100).round(2)

emotion_stats

,Emotion,Records,Percentage
0,informative,36440,39.75
1,neutral,19232,20.98
2,reflective,6741,7.35
3,supportive,5251,5.73
4,calm,4978,5.43
5,curious,3330,3.63
6,cautious,2771,3.02
7,serious,1644,1.79
8,reassuring,1502,1.64
9,concerned,1044,1.14


In [19]:
game_context_stats = df["labels.game_context"].value_counts().reset_index()
game_context_stats.columns = ["Game Context", "Records"]
game_context_stats["Percentage"] = (game_context_stats["Records"] / len(df) * 100).round(2)

game_context_stats

,Game Context,Records,Percentage
0,tutorial,28277,30.84
1,social_interaction,8869,9.67
2,mission_briefing,8711,9.50
3,debriefing,8056,8.79
4,emotional_support,4799,5.23
5,decision_point,4249,4.63
6,navigation,4230,4.61
7,safety_warning,3475,3.79
8,exploration,2322,2.53
9,quest_progress,2137,2.33


In [20]:
difficulty_stats = df["labels.difficulty_level"].value_counts().reset_index()
difficulty_stats.columns = ["Difficulty Level", "Records"]
difficulty_stats["Percentage"] = (difficulty_stats["Records"] / len(df) * 100).round(2)

difficulty_stats

,Difficulty Level,Records,Percentage
0,easy,78910,86.08
1,medium,8687,9.48
2,very_easy,4058,4.43
3,hard,19,0.02
4,adaptive,1,0.00


In [21]:
length_summary = pd.DataFrame({
    "Metric": [
        "Mean question length",
        "Median question length",
        "Std question length",
        "Min question length",
        "Max question length",
        "Mean answer length",
        "Median answer length",
        "Std answer length",
        "Min answer length",
        "Max answer length",
        "Mean dialogue length",
        "Median dialogue length",
        "Std dialogue length",
        "Min dialogue length",
        "Max dialogue length"
    ],
    "Turkish": [
        df["question_tr_token_len"].mean(),
        df["question_tr_token_len"].median(),
        df["question_tr_token_len"].std(),
        df["question_tr_token_len"].min(),
        df["question_tr_token_len"].max(),
        df["answer_tr_token_len"].mean(),
        df["answer_tr_token_len"].median(),
        df["answer_tr_token_len"].std(),
        df["answer_tr_token_len"].min(),
        df["answer_tr_token_len"].max(),
        df["dialogue_tr_token_len"].mean(),
        df["dialogue_tr_token_len"].median(),
        df["dialogue_tr_token_len"].std(),
        df["dialogue_tr_token_len"].min(),
        df["dialogue_tr_token_len"].max()
    ],
    "English": [
        df["question_en_token_len"].mean(),
        df["question_en_token_len"].median(),
        df["question_en_token_len"].std(),
        df["question_en_token_len"].min(),
        df["question_en_token_len"].max(),
        df["answer_en_token_len"].mean(),
        df["answer_en_token_len"].median(),
        df["answer_en_token_len"].std(),
        df["answer_en_token_len"].min(),
        df["answer_en_token_len"].max(),
        df["dialogue_en_token_len"].mean(),
        df["dialogue_en_token_len"].median(),
        df["dialogue_en_token_len"].std(),
        df["dialogue_en_token_len"].min(),
        df["dialogue_en_token_len"].max()
    ]
})

length_summary = length_summary.round(2)
length_summary

,Metric,Turkish,English
0,Mean question length,6.13,8.62
1,Median question length,6.00,8.00
2,Std question length,1.76,2.89
3,Min question length,1.00,1.00
4,Max question length,21.00,33.00
5,Mean answer length,5.40,7.92
6,Median answer length,5.00,7.00
7,Std answer length,2.21,3.38
8,Min answer length,1.00,1.00
9,Max answer length,28.00,36.00


In [22]:
import re

def tokenize_text(text):
    if pd.isna(text):
        return []
    return re.findall(r"\b\w+\b", str(text).lower())

tr_tokens = []
en_tokens = []

for text in df["dialogue.question_tr"].tolist() + df["dialogue.answer_tr"].tolist():
    tr_tokens.extend(tokenize_text(text))

for text in df["dialogue.question_en"].tolist() + df["dialogue.answer_en"].tolist():
    en_tokens.extend(tokenize_text(text))

vocab_stats = pd.DataFrame({
    "Metric": [
        "Total Turkish tokens",
        "Unique Turkish tokens",
        "Total English tokens",
        "Unique English tokens"
    ],
    "Value": [
        len(tr_tokens),
        len(set(tr_tokens)),
        len(en_tokens),
        len(set(en_tokens))
    ]
})

vocab_stats

,Metric,Value
0,Total Turkish tokens,1064137
1,Unique Turkish tokens,50530
2,Total English tokens,1516108
3,Unique English tokens,16168


In [23]:
type_token_stats = pd.DataFrame({
    "Language": ["Turkish", "English"],
    "Total Tokens": [len(tr_tokens), len(en_tokens)],
    "Unique Tokens": [len(set(tr_tokens)), len(set(en_tokens))],
    "Type-Token Ratio": [
        len(set(tr_tokens)) / len(tr_tokens),
        len(set(en_tokens)) / len(en_tokens)
    ]
})

type_token_stats["Type-Token Ratio"] = type_token_stats["Type-Token Ratio"].round(4)
type_token_stats

,Language,Total Tokens,Unique Tokens,Type-Token Ratio
0,Turkish,1064137,50530,0.0475
1,English,1516108,16168,0.0107


In [24]:
intent_game_context_stats = df.groupby(
    ["labels.intent", "labels.game_context"]
).size().reset_index(name="Records")

intent_game_context_stats = intent_game_context_stats.sort_values(
    by="Records",
    ascending=False
)

intent_game_context_stats.head(30)

,labels.intent,labels.game_context,Records
28,ACADEMIC_OVERVIEW,tutorial,13686
142,CONCEPT_EXPLAIN,tutorial,8306
438,REFLECTION,debriefing,3198
139,CONCEPT_EXPLAIN,social_interaction,2658
420,PROCESS_GUIDE,mission_briefing,2602
256,EVALUATE,mission_briefing,2397
4,ACADEMIC_OVERVIEW,debriefing,2261
24,ACADEMIC_OVERVIEW,social_interaction,1981
16,ACADEMIC_OVERVIEW,navigation,1955
23,ACADEMIC_OVERVIEW,safety_warning,1915


In [25]:
emotion_context_stats = df.groupby(
    ["labels.emotion", "labels.game_context"]
).size().reset_index(name="Records").sort_values(by="Records", ascending=False)

emotion_context_stats.head(30)

,labels.emotion,labels.game_context,Records
292,informative,tutorial,19894
288,informative,social_interaction,3609
358,neutral,mission_briefing,3491
278,informative,mission_briefing,3229
372,neutral,tutorial,3150
461,reflective,debriefing,3143
566,supportive,tutorial,2413
280,informative,navigation,1958
348,neutral,debriefing,1824
79,cautious,tutorial,1498


In [26]:
role_intent_stats = df.groupby(
    ["npc.role", "labels.intent"]
).size().reset_index(name="Records").sort_values(by="Records", ascending=False)

role_intent_stats.head(30)

,npc.role,labels.intent,Records
282,travel_advisor_npc,CONCEPT_EXPLAIN,6448
299,travel_advisor_npc,PROCESS_GUIDE,3484
130,information_expert,ACADEMIC_OVERVIEW,2772
291,travel_advisor_npc,EVALUATE,2643
278,travel_advisor_npc,AWARENESS_BUILD,2230
90,dialogue_reflection_npc,REFLECTION,1900
11,alparslan_npc,ACADEMIC_OVERVIEW,1480
59,dialogue_entertainment_npc,DECISION_SUPPORT,1413
285,travel_advisor_npc,DECISION_SUPPORT,1400
74,dialogue_entertainment_npc,PROCESS_GUIDE,1359


In [27]:
domain_intent_matrix = pd.crosstab(
    df["domain"],
    df["labels.intent"]
)

domain_intent_matrix

labels.intent,ACADEMIC_OVERVIEW,APPLICATION,AWARENESS_BUILD,CAUSE_EFFECT_EXPLAIN,CLASSIFICATION,COMMUNICATION_SUPPORT,COMPARISON,CONCEPT_EXPLAIN,DATA_ANALYZE,DATA_COLLECT,...,PROCESS_EXPLAIN,PROCESS_GUIDE,REFLECTION,RISK_MANAGEMENT,ROLE_EXPLAIN,RULE_EXPLAIN,SAFETY_GUIDANCE,SCI_PROCESS,SKILL_EXPLAIN,SOCIAL_INTERACTION
domain,,,,,,,,,,,,,,,,,,,,,
academic,0,0,0,0,0,0,0,646,0,0,...,0,3,0,1,0,0,0,0,0,0
artificial_intelligence,600,0,1,0,4,0,2,499,0,0,...,0,0,0,0,0,0,0,0,0,0
assessment,0,0,0,0,0,0,0,534,0,0,...,3,0,0,0,0,0,0,0,0,0
biology,0,0,0,0,0,0,0,382,0,0,...,0,0,0,0,0,0,0,0,0,0
chemistry,0,0,0,0,0,0,0,471,2,0,...,15,4,0,0,0,0,3,0,0,0
coding,1,0,0,55,0,0,0,183,0,0,...,59,0,0,46,0,0,0,0,0,0
counseling,750,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
crisis,1200,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
data_analysis,0,0,0,0,0,0,0,505,0,0,...,9,0,0,0,0,0,0,0,0,0


In [28]:
domain_intent_matrix_pct = pd.crosstab(
    df["domain"],
    df["labels.intent"],
    normalize="index"
) * 100

domain_intent_matrix_pct = domain_intent_matrix_pct.round(2)
domain_intent_matrix_pct

labels.intent,ACADEMIC_OVERVIEW,APPLICATION,AWARENESS_BUILD,CAUSE_EFFECT_EXPLAIN,CLASSIFICATION,COMMUNICATION_SUPPORT,COMPARISON,CONCEPT_EXPLAIN,DATA_ANALYZE,DATA_COLLECT,...,PROCESS_EXPLAIN,PROCESS_GUIDE,REFLECTION,RISK_MANAGEMENT,ROLE_EXPLAIN,RULE_EXPLAIN,SAFETY_GUIDANCE,SCI_PROCESS,SKILL_EXPLAIN,SOCIAL_INTERACTION
domain,,,,,,,,,,,,,,,,,,,,,
academic,0.00,0.00,0.00,0.00,0.00,0.00,0.00,99.38,0.00,0.00,...,0.00,0.46,0.00,0.15,0.00,0.00,0.00,0.00,0.00,0.00
artificial_intelligence,54.15,0.00,0.09,0.00,0.36,0.00,0.18,45.04,0.00,0.00,...,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
assessment,0.00,0.00,0.00,0.00,0.00,0.00,0.00,99.44,0.00,0.00,...,0.56,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
biology,0.00,0.00,0.00,0.00,0.00,0.00,0.00,83.59,0.00,0.00,...,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
chemistry,0.00,0.00,0.00,0.00,0.00,0.00,0.00,95.15,0.40,0.00,...,3.03,0.81,0.00,0.00,0.00,0.00,0.61,0.00,0.00,0.00
coding,0.22,0.00,0.00,12.01,0.00,0.00,0.00,39.96,0.00,0.00,...,12.88,0.00,0.00,10.04,0.00,0.00,0.00,0.00,0.00,0.00
counseling,100.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,...,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
crisis,100.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,...,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
data_analysis,0.00,0.00,0.00,0.00,0.00,0.00,0.00,97.87,0.00,0.00,...,1.74,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00


In [29]:
domain_game_context_matrix = pd.crosstab(
    df["domain"],
    df["labels.game_context"]
)

domain_game_context_matrix

labels.game_context,character_relationship,combat,combat_preparation,crafting,debriefing,decision_point,dialogue_choice,emergency,emotional_support,exploration,...,quest_progress,quest_start,role_play_training,safety_warning,social_interaction,story_exposition,success_feedback,trading,tutorial,world_event
domain,,,,,,,,,,,,,,,,,,,,,
academic,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,650,0
artificial_intelligence,0,0,0,0,0,49,0,51,0,0,...,0,0,0,150,0,0,0,0,558,0
assessment,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,537,0
biology,0,0,0,0,1,0,0,0,0,0,...,0,17,0,0,0,0,0,0,439,0
chemistry,0,0,0,0,49,0,0,0,0,0,...,0,0,0,5,0,0,0,0,441,0
coding,0,0,0,0,0,0,0,0,0,0,...,0,43,0,0,0,0,0,0,415,0
counseling,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,148,0,0,0,50,0
crisis,200,110,0,0,100,0,0,498,0,0,...,0,0,0,198,0,0,0,0,0,0
data_analysis,0,0,0,0,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,515,0


In [30]:
duplicate_stats = pd.DataFrame({
    "Metric": [
        "Unique Turkish questions",
        "Duplicate Turkish questions",
        "Unique English questions",
        "Duplicate English questions",
        "Unique Turkish answers",
        "Duplicate Turkish answers",
        "Unique English answers",
        "Duplicate English answers"
    ],
    "Value": [
        df["dialogue.question_tr"].nunique(),
        len(df) - df["dialogue.question_tr"].nunique(),
        df["dialogue.question_en"].nunique(),
        len(df) - df["dialogue.question_en"].nunique(),
        df["dialogue.answer_tr"].nunique(),
        len(df) - df["dialogue.answer_tr"].nunique(),
        df["dialogue.answer_en"].nunique(),
        len(df) - df["dialogue.answer_en"].nunique()
    ]
})

duplicate_stats

,Metric,Value
0,Unique Turkish questions,91271
1,Duplicate Turkish questions,404
2,Unique English questions,90598
3,Duplicate English questions,1077
4,Unique Turkish answers,87173
5,Duplicate Turkish answers,4502
6,Unique English answers,86379
7,Duplicate English answers,5296


In [31]:
df["answer_question_ratio_tr"] = df["answer_tr_token_len"] / df["question_tr_token_len"].replace(0, 1)
df["answer_question_ratio_en"] = df["answer_en_token_len"] / df["question_en_token_len"].replace(0, 1)

ratio_stats = pd.DataFrame({
    "Metric": [
        "Mean answer/question ratio TR",
        "Median answer/question ratio TR",
        "Mean answer/question ratio EN",
        "Median answer/question ratio EN"
    ],
    "Value": [
        df["answer_question_ratio_tr"].mean(),
        df["answer_question_ratio_tr"].median(),
        df["answer_question_ratio_en"].mean(),
        df["answer_question_ratio_en"].median()
    ]
})

ratio_stats = ratio_stats.round(2)
ratio_stats

,Metric,Value
0,Mean answer/question ratio TR,0.93
1,Median answer/question ratio TR,0.83
2,Mean answer/question ratio EN,1.00
3,Median answer/question ratio EN,0.89


In [32]:
import numpy as np

def shannon_entropy(series):
    counts = series.value_counts()
    probs = counts / counts.sum()
    entropy = -np.sum(probs * np.log2(probs))
    max_entropy = np.log2(len(counts)) if len(counts) > 1 else 1
    normalized_entropy = entropy / max_entropy if max_entropy > 0 else 0
    return entropy, normalized_entropy

balance_rows = []

for col in [
    "domain",
    "npc.role",
    "labels.intent",
    "labels.player_intent",
    "labels.emotion",
    "labels.game_context",
    "labels.difficulty_level"
]:
    entropy, norm_entropy = shannon_entropy(df[col])
    balance_rows.append({
        "Label Group": col,
        "Number of Classes": df[col].nunique(),
        "Entropy": round(entropy, 4),
        "Normalized Entropy": round(norm_entropy, 4)
    })

balance_stats = pd.DataFrame(balance_rows)
balance_stats

,Label Group,Number of Classes,Entropy,Normalized Entropy
0,domain,32,3.2569,0.6514
1,npc.role,76,5.2184,0.8352
2,labels.intent,37,3.4781,0.6677
3,labels.player_intent,25,2.6969,0.5808
4,labels.emotion,40,3.0533,0.5737
5,labels.game_context,30,3.7305,0.7603
6,labels.difficulty_level,5,0.7101,0.3058


In [33]:
from collections import Counter

tr_counter = Counter(tr_tokens)
en_counter = Counter(en_tokens)

top_tr_words = pd.DataFrame(
    tr_counter.most_common(50),
    columns=["Turkish Token", "Frequency"]
)

top_en_words = pd.DataFrame(
    en_counter.most_common(50),
    columns=["English Token", "Frequency"]
)

top_tr_words.head(20), top_en_words.head(20)

(   Turkish Token  Frequency
 0             ve      27849
 1          nasıl      16377
 2             mi      14310
 3          neden      13551
 4            bir      12279
 5             ne      10823
 6             mı      10814
 7           için       9992
 8              i       7862
 9         sağlar       7325
 10       etkiler       6175
 11         hangi       6147
 12          daha       5904
 13          evet       5716
 14         nedir       5440
 15          olur       4274
 16         zaman       4106
 17            bu       3737
 18            mu       3330
 19       artırır       3223,
    English Token  Frequency
 0            the      97838
 1             it      44454
 2             is      41651
 3             of      34576
 4              a      34559
 5             to      33166
 6            and      31960
 7           what      25092
 8             in      23648
 9            you      22265
 10          does      17917
 11           how      16984
 12          

In [34]:
domain_vocab_rows = []

for domain, group in df.groupby("domain"):
    domain_tr_tokens = []
    domain_en_tokens = []

    for text in group["dialogue.question_tr"].tolist() + group["dialogue.answer_tr"].tolist():
        domain_tr_tokens.extend(tokenize_text(text))

    for text in group["dialogue.question_en"].tolist() + group["dialogue.answer_en"].tolist():
        domain_en_tokens.extend(tokenize_text(text))

    domain_vocab_rows.append({
        "Domain": domain,
        "Records": len(group),
        "Total Turkish Tokens": len(domain_tr_tokens),
        "Unique Turkish Tokens": len(set(domain_tr_tokens)),
        "Total English Tokens": len(domain_en_tokens),
        "Unique English Tokens": len(set(domain_en_tokens))
    })

domain_vocab_stats = pd.DataFrame(domain_vocab_rows)
domain_vocab_stats

,Domain,Records,Total Turkish Tokens,Unique Turkish Tokens,Total English Tokens,Unique English Tokens
0,academic,650,6069,1877,9492,1414
1,artificial_intelligence,1108,12119,2734,16793,1939
2,assessment,537,5034,1290,6994,1022
3,biology,457,4793,1491,7248,1089
4,chemistry,495,4528,1401,6918,1063
5,coding,458,4824,1624,7034,1194
6,counseling,750,8231,1873,11359,1429
7,crisis,1200,13299,2208,18676,1684
8,data_analysis,516,5253,1266,7301,1029
9,design_production,516,6996,1600,9526,1232


In [35]:
intent_answer_length = df.groupby("labels.intent").agg(
    records=("id", "count"),
    avg_answer_tr_len=("answer_tr_token_len", "mean"),
    avg_answer_en_len=("answer_en_token_len", "mean"),
    avg_dialogue_tr_len=("dialogue_tr_token_len", "mean"),
    avg_dialogue_en_len=("dialogue_en_token_len", "mean")
).reset_index()

intent_answer_length = intent_answer_length.sort_values(
    by="records",
    ascending=False
).round(2)

intent_answer_length.head(30)

,labels.intent,records,avg_answer_tr_len,avg_answer_en_len,avg_dialogue_tr_len,avg_dialogue_en_len
0,ACADEMIC_OVERVIEW,27591,5.75,8.38,11.80,16.69
7,CONCEPT_EXPLAIN,17932,5.37,8.17,11.04,16.17
28,PROCESS_GUIDE,8504,5.18,7.55,11.67,16.87
29,REFLECTION,5505,5.06,7.30,11.57,16.74
21,KNOWLEDGE_CHECK,4262,5.09,7.30,11.18,15.79
2,AWARENESS_BUILD,4228,4.89,7.06,10.91,15.54
16,EVALUATE,4070,5.26,7.96,11.94,17.68
10,DECISION_SUPPORT,3918,5.15,7.34,11.69,16.49
23,LEARNING_SUPPORT,3866,6.36,9.21,13.02,18.58
9,DATA_COLLECT,1149,5.59,8.50,11.17,16.17


In [37]:
excel_path = "npc_dataset_descriptive_statistics.xlsx"

with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
    basic_stats.to_excel(writer, sheet_name="Basic_Stats", index=False)
    domain_stats.to_excel(writer, sheet_name="Domain_Distribution", index=False)
    npc_role_stats.to_excel(writer, sheet_name="NPC_Role_Distribution", index=False)
    intent_stats.to_excel(writer, sheet_name="Intent_Distribution", index=False)
    player_intent_stats.to_excel(writer, sheet_name="Player_Intent", index=False)
    emotion_stats.to_excel(writer, sheet_name="Emotion_Distribution", index=False)
    game_context_stats.to_excel(writer, sheet_name="Game_Context", index=False)
    difficulty_stats.to_excel(writer, sheet_name="Difficulty", index=False)
    length_summary.to_excel(writer, sheet_name="Length_Summary", index=False)
    vocab_stats.to_excel(writer, sheet_name="Vocabulary_Stats", index=False)
    type_token_stats.to_excel(writer, sheet_name="Type_Token_Ratio", index=False)
    duplicate_stats.to_excel(writer, sheet_name="Duplicate_Stats", index=False)
    ratio_stats.to_excel(writer, sheet_name="Answer_Question_Ratio", index=False)
    balance_stats.to_excel(writer, sheet_name="Balance_Entropy", index=False)
    domain_vocab_stats.to_excel(writer, sheet_name="Domain_Vocab", index=False)
    domain_intent_matrix.to_excel(writer, sheet_name="Domain_Intent_Matrix")
    domain_game_context_matrix.to_excel(writer, sheet_name="Domain_Context_Matrix")

print("Excel dosyası oluşturuldu:", excel_path)

Excel dosyası oluşturuldu: npc_dataset_descriptive_statistics.xlsx


In [38]:
from google.colab import files
files.download("npc_dataset_descriptive_statistics.xlsx")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>